# Pipeline of execution: 

* Run `resample.py`
* Run `features_volumetric.py`
* Run `features_radiomic.py`
* Run `features_deltas_displacement.py`
* Run `features_displacement.py`

# Atributos unitários

In [22]:
# # Leitura radiomicos e volumetricos

# import pandas as pd

# ab = "abordagem_1_sMCI_pMCI"

# vol_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_volumetric.csv"

# rad_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_radiomic.csv"

# vol_df = pd.read_csv(vol_path)
# rad_df = pd.read_csv(rad_path)


In [23]:
# # Normalizacao pelo ICV (mask_mm3 global)

# # ICV = volume total da mascara do encefalo (linha __global__ em features_volumetric)
# # Vamos anexar o ICV por ID_IMG e normalizar apenas as features dependentes de tamanho.

# import numpy as np

# # 1) ICV por ID_IMG
# vol_df["ID_IMG"] = vol_df["ID_IMG"].astype(str).str.strip()

# icv_df = (
#     vol_df.loc[vol_df["roi"].astype(str) == "__global__", ["ID_IMG", "mask_mm3"]]
#     .drop_duplicates(subset=["ID_IMG"], keep="last")
#     .rename(columns={"mask_mm3": "ICV_mask_mm3"})
# )

# icv_df["ICV_mask_mm3"] = pd.to_numeric(icv_df["ICV_mask_mm3"], errors="coerce")

# # 2) Merge com radiomicos
# rad_df["ID_IMG"] = rad_df["ID_IMG"].astype(str).str.strip()
# merged = rad_df.merge(icv_df, on="ID_IMG", how="left", validate="many_to_one")

# missing_icv = int(merged["ICV_mask_mm3"].isna().sum())
# if missing_icv:
#     raise ValueError(
#         f"ICV ausente para {missing_icv} linhas radiomicas. "
#         "Verifique se todos os ID_IMG do radiomico existem no volumetrico (linha __global__)."
#     )

# # 3) Features a normalizar (dependentes de tamanho)
# cols_to_norm = [
#     # first-order
#     "original_firstorder_Energy",
#     "original_firstorder_TotalEnergy",
#     # shape (absolutas)
#     "original_shape_MeshVolume",
#     "original_shape_VoxelVolume",
#     "original_shape_SurfaceArea",
#     "original_shape_LeastAxisLength",
#     "original_shape_MajorAxisLength",
#     "original_shape_MinorAxisLength",
#     "original_shape_Maximum2DDiameterColumn",
#     "original_shape_Maximum2DDiameterRow",
#     "original_shape_Maximum2DDiameterSlice",
#     "original_shape_Maximum3DDiameter",
# ]

# # mantém só as que existem no arquivo (robustez)
# cols_to_norm = [c for c in cols_to_norm if c in merged.columns]

# # garante numérico
# for c in cols_to_norm:
#     merged[c] = pd.to_numeric(merged[c], errors="coerce")

# # 4) Normaliza pelo ICV
# # Observacao: algumas features (p.ex. area) ficam com unidade relativa a volume;
# # aqui seguimos a definicao pedida: dividir pelo volume total do encefalo.
# merged[cols_to_norm] = merged[cols_to_norm].div(merged["ICV_mask_mm3"], axis=0)

# # 5) CSV final: sem colunas "brutas" não-normalizadas adicionais (mantemos só a versão normalizada)
# # (as colunas normalizadas sobrescrevem as originais, então basta salvar o merged)
# out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"
# merged.to_csv(out_path, index=False)

# out_path


In [24]:
# # Leitura dados técnicos e inserção dos equipamentos na planilha de atributos radiomicos

# import pandas as pd

# tech_path = f"/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv"

# radiomics_merged_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"

# tech_df = pd.read_csv(tech_path)
# radiomics_merged_df = pd.read_csv(radiomics_merged_path)

In [25]:
# # Inserção de variáveis técnicas (por ID_IMG)

# import numpy as np
# import pandas as pd

# # A planilha "merge" será o radiomics_merged_df enriquecido com informações técnicas/clínicas.
# # (Se você preferir outro nome, basta trocar a variável aqui.)
# merge = radiomics_merged_df.copy()

# # 1) Merge por ID_IMG trazendo as colunas necessárias
# cols_needed = [
#     "ID_IMG",
#     "ID_PT",
#     "SEX",
#     "DIAG",
#     "MRI_DATE",
#     "FIELD_STRENGTH",
#     "SLICE_THICKNESS",
#     "MANUFACTURER",
#     "MFG_MODEL",
# ]

# tech_sub = tech_df.loc[:, cols_needed].copy()
# tech_sub["ID_IMG"] = tech_sub["ID_IMG"].astype(str).str.strip()
# tech_sub = tech_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

# merge["ID_IMG"] = merge["ID_IMG"].astype(str).str.strip()
# merge = merge.merge(tech_sub, on="ID_IMG", how="left", validate="many_to_one")

# # 2) Define batch (scanner) a partir das variáveis do equipamento
# merge["batch"] = (
#     merge[["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH", "SLICE_THICKNESS"]]
#     .astype(str)
#     .agg("_".join, axis=1)
# )

# missing_any = int(merge[cols_needed[1:]].isna().any(axis=1).sum())
# print(f"Linhas sem alguma info técnica/clínica necessária (após merge): {missing_any}")



# Merge atributos unitários

In [26]:
# import pandas as pd

# rad_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"
# disp_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_displacement_unitary.csv"
# out_all_unit = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_features_unitary_merge.csv"

# rad = pd.read_csv(rad_path)
# print(rad.shape)
# disp = pd.read_csv(disp_path)
# print(disp.shape)

# # (Opcional, mas recomendado) padroniza tipos das chaves
# for df in (rad, disp):
#     df["ID_IMG"] = df["ID_IMG"].astype(str).str.strip()
#     df["roi"] = df["roi"].astype(str).str.strip()
#     df["side"] = df["side"].astype(str).str.strip()
#     df["label"] = df["label"].astype(str).str.strip()

# all_unit = rad.merge(
#     disp,
#     on=["ID_IMG", "roi", "side", "label"],
#     how="left",
#     validate="one_to_one",  # se der erro aqui, há duplicatas em algum arquivo
# )

# all_unit.to_csv(out_all_unit, index=False)
# print(all_unit.shape)

In [27]:
# # Inserir os atributos por triplet
# # Inserir os atributos por triplet (LONG, similar ao all_delta_features.csv)
# #
# # Entrada:
# # - cj_data_abordagem_teste.txt (define conjuntos i1/i2/i3 por paciente/combinação)
# # - all_features_unitary_merge.csv (atributos unitários por ID_IMG x ROI)
# #
# # Saída:
# # - all_unitary_features_triplets_long.csv (1 linha por triplet + ROI + imagem de referência)

# import os
# import numpy as np
# import pandas as pd

# cj_path = f"/mnt/study-data/pgirardi/graphs/cj_data_{ab}.txt"
# unit_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_features_unitary_merge.csv"
# out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_unitary_features.csv"

# # --- lê conjuntos (long) ---
# cj = pd.read_csv(cj_path)

# # normaliza tipos/chaves
# cj["ID_IMG"] = cj["ID_IMG"].astype(str).str.strip()
# cj["ID_PT"] = cj["ID_PT"].astype(str).str.strip()

# # datas
# cj["MRI_DATE"] = pd.to_datetime(cj["MRI_DATE"], errors="coerce")

# # ordenação (suporta múltiplos triplets por grupo)
# sort_cols = ["ID_PT", "COMBINATION_NUMBER", "MRI_DATE"]
# if "TIME_PROG" in cj.columns:
#     sort_cols.append("TIME_PROG")
# sort_cols.append("ID_IMG")

# cj = cj.sort_values(sort_cols)

# # índice do timepoint dentro do grupo e decomposição em TRIPLET_IDX e posição 0/1/2
# pos_in_group = cj.groupby(["ID_PT", "COMBINATION_NUMBER"]).cumcount()
# cj["TRIPLET_IDX"] = (pos_in_group // 3).astype(int)
# cj["_pos"] = (pos_in_group % 3).astype(int)

# # mantém apenas triplets completos (pos 0,1,2)
# complete = cj.groupby(["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"])["_pos"].nunique()
# complete = complete[complete == 3].index
# cj = cj.set_index(["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]).loc[complete].reset_index()

# # pivot para obter ID_IMG_i1/i2/i3 e MRI_DATE_i1/i2/i3
# wide_img = (
#     cj.pivot_table(
#         index=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"],
#         columns="_pos",
#         values="ID_IMG",
#         aggfunc="first",
#     )
#     .rename(columns={0: "ID_IMG_i1", 1: "ID_IMG_i2", 2: "ID_IMG_i3"})
#     .reset_index()
# )

# wide_date = (
#     cj.pivot_table(
#         index=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"],
#         columns="_pos",
#         values="MRI_DATE",
#         aggfunc="first",
#     )
#     .rename(columns={0: "MRI_DATE_i1", 1: "MRI_DATE_i2", 2: "MRI_DATE_i3"})
#     .reset_index()
# )

# trip = wide_img.merge(
#     wide_date,
#     on=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"],
#     how="inner",
#     validate="one_to_one",
# )

# # tempos em meses (como no bloco de deltas)
# DAYS_PER_MONTH = 365.2425 / 12.0
# trip["t12"] = ((trip["MRI_DATE_i2"] - trip["MRI_DATE_i1"]).dt.total_seconds() / 86400.0) / DAYS_PER_MONTH
# trip["t13"] = ((trip["MRI_DATE_i3"] - trip["MRI_DATE_i1"]).dt.total_seconds() / 86400.0) / DAYS_PER_MONTH
# trip["t23"] = ((trip["MRI_DATE_i3"] - trip["MRI_DATE_i2"]).dt.total_seconds() / 86400.0) / DAYS_PER_MONTH

# trip = trip.drop(columns=["MRI_DATE_i1", "MRI_DATE_i2", "MRI_DATE_i3"])

# # --- LONG por imagem (i1/i2/i3) ---
# long_imgs = trip.melt(
#     id_vars=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "t12", "t13", "t23"],
#     value_vars=["ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3"],
#     var_name="pair",
#     value_name="ID_IMG_ref",
# )

# # no arquivo unitário: pair = número da imagem no conjunto (1/2/3)
# long_imgs["pair"] = long_imgs["pair"].replace({"ID_IMG_i1": 1, "ID_IMG_i2": 2, "ID_IMG_i3": 3}).astype(int)
# long_imgs["ID_IMG_ref"] = long_imgs["ID_IMG_ref"].astype(str).str.strip()

# # recoloca ID_IMG_i1/i2/i3 (para ficar com o mesmo prefixo do all_delta_features.csv)
# long_imgs = long_imgs.merge(
#     trip[["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3"]],
#     on=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"],
#     how="left",
#     validate="many_to_one",
# )

# # --- lê atributos unitários (por imagem x ROI) ---
# unit = pd.read_csv(unit_path)
# for c in ("ID_IMG", "roi", "side", "label"):
#     unit[c] = unit[c].astype(str).str.strip()

# roi_keys = ["ID_IMG", "roi", "side", "label"]
# unit = unit.drop_duplicates(subset=roi_keys, keep="last")

# # evita colisão de colunas (pandas criaria ID_PT_x/ID_PT_y, DIAG_x/DIAG_y, etc.)
# # vamos manter metadados/covariáveis vindos do cj/adnimerged
# _reserved = {
#     "ID_PT",
#     "COMBINATION_NUMBER",
#     "TRIPLET_IDX",
#     "pair",
#     "ID_IMG_i1",
#     "ID_IMG_i2",
#     "ID_IMG_i3",
#     "t12",
#     "t13",
#     "t23",
#     "GROUP",
#     "SEX",
#     "DIAG",
#     "AGE",
#     "TIME_PROG",
#     "FIELD_STRENGTH",
#     "SLICE_THICKNESS",
#     "MANUFACTURER",
#     "MFG_MODEL",
#     "MRI_DATE",
#     "batch",
#     "ID_IMG_ref",
# }
# _drop_from_unit = [c for c in unit.columns if c in _reserved and c not in roi_keys and c != "ID_IMG"]
# if _drop_from_unit:
#     unit = unit.drop(columns=_drop_from_unit)

# out = long_imgs.merge(
#     unit.rename(columns={"ID_IMG": "ID_IMG_ref"}),
#     on=["ID_IMG_ref"],
#     how="inner",
#     validate="many_to_many",
# )

# # --- covariáveis e técnicos por ID_IMG_ref (compatível com all_delta_features.csv) ---
# # 1) tenta do próprio cj
# img_cov_cols = [c for c in ["ID_IMG", "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG"] if c in cj.columns]
# if img_cov_cols:
#     img_cov = cj.loc[:, img_cov_cols].copy()
#     img_cov["ID_IMG"] = img_cov["ID_IMG"].astype(str).str.strip()
#     img_cov = img_cov.drop_duplicates(subset=["ID_IMG"], keep="last")
#     out = out.merge(
#         img_cov.rename(columns={"ID_IMG": "ID_IMG_ref"}),
#         on="ID_IMG_ref",
#         how="left",
#         validate="many_to_one",
#     )

# # 2) complementa com adnimerged.csv
# tech_path = "/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv"
# if os.path.exists(tech_path):
#     tech_df = pd.read_csv(tech_path)
#     tech_df["ID_IMG"] = tech_df["ID_IMG"].astype(str).str.strip()
#     tech_keep = [
#         c
#         for c in [
#             "ID_IMG",
#             "GROUP",
#             "SEX",
#             "DIAG",
#             "AGE",
#             "TIME_PROG",
#             "FIELD_STRENGTH",
#             "SLICE_THICKNESS",
#             "MANUFACTURER",
#             "MFG_MODEL",
#         ]
#         if c in tech_df.columns
#     ]
#     tech_sub = tech_df.loc[:, tech_keep].drop_duplicates(subset=["ID_IMG"], keep="last")
#     out = out.merge(
#         tech_sub.rename(columns={"ID_IMG": "ID_IMG_ref"}),
#         on="ID_IMG_ref",
#         how="left",
#         validate="many_to_one",
#         suffixes=("", "_tech"),
#     )

#     for c in [
#         "GROUP",
#         "SEX",
#         "DIAG",
#         "AGE",
#         "TIME_PROG",
#         "FIELD_STRENGTH",
#         "SLICE_THICKNESS",
#         "MANUFACTURER",
#         "MFG_MODEL",
#     ]:
#         c_tech = f"{c}_tech"
#         if c_tech in out.columns:
#             if c in out.columns:
#                 out[c] = out[c].where(out[c].notna(), out[c_tech])
#             else:
#                 out[c] = out[c_tech]
#             out = out.drop(columns=[c_tech])

# tech_for_batch = [c for c in ["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH", "SLICE_THICKNESS"] if c in out.columns]
# if tech_for_batch:
#     out["batch"] = out[tech_for_batch].astype(str).agg("_".join, axis=1)
# else:
#     out["batch"] = np.nan

# # ordena colunas seguindo a ordem do all_delta_features.csv
# delta_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features.csv"
# delta_cols = pd.read_csv(delta_path, nrows=0).columns.tolist() if os.path.exists(delta_path) else []

# # garante que o prefixo mínimo exista (mesmo se delta não estiver presente)
# required_prefix = [
#     "ID_PT",
#     "COMBINATION_NUMBER",
#     "TRIPLET_IDX",
#     "pair",
#     "ID_IMG_i1",
#     "ID_IMG_i2",
#     "ID_IMG_i3",
#     "roi",
#     "side",
#     "label",
#     "t12",
#     "t13",
#     "t23",
#     "GROUP",
#     "SEX",
#     "DIAG",
#     "AGE",
#     "TIME_PROG",
#     "ID_IMG_ref",
#     "FIELD_STRENGTH",
#     "SLICE_THICKNESS",
#     "MANUFACTURER",
#     "MFG_MODEL",
#     "batch",
# ]
# for c in required_prefix:
#     if c not in out.columns:
#         out[c] = np.nan

# # ordem: (i) colunas do delta que existem aqui (respeita sequência do delta)
# #      (ii) demais colunas restantes
# ordered = [c for c in delta_cols if c in out.columns] if delta_cols else []
# if not ordered:
#     ordered = [c for c in required_prefix if c in out.columns]
# ordered += [c for c in out.columns if c not in ordered]

# # força ref_tag logo após ID_IMG_i3 (se existir)
# if "ref_tag" in ordered and "ID_IMG_i3" in ordered:
#     ordered = [c for c in ordered if c != "ref_tag"]
#     idx = ordered.index("ID_IMG_i3") + 1
#     ordered.insert(idx, "ref_tag")

# out = out.loc[:, ordered]

# out.to_csv(out_path, index=False)
# print("Salvo:", out_path)
# print("shape:", out.shape)


# Delta atributos

In [28]:
# # Merge: deslocamento + deltas radiômicos

# import pandas as pd
# import numpy as np

# rad_delta_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/deltas_radiomics.csv"
# disp_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/deltas_displacement_{ab}.csv"

# rad_delta_df = pd.read_csv(rad_delta_path)
# disp_df2 = pd.read_csv(disp_path)

# # chaves comuns esperadas
# keys = [
#     "ID_PT",
#     "COMBINATION_NUMBER",
#     "TRIPLET_IDX",
#     "pair",
#     "ID_IMG_i1",
#     "ID_IMG_i2",
#     "ID_IMG_i3",
#     "roi",
#     "side",
#     "label",
# ]

# missing_rad = [c for c in keys if c not in rad_delta_df.columns]
# missing_disp = [c for c in keys if c not in disp_df2.columns]
# if missing_rad:
#     raise ValueError(f"deltas_radiomics.csv sem colunas-chave: {missing_rad}")
# if missing_disp:
#     raise ValueError(f"displacement_with_strain.csv sem colunas-chave: {missing_disp}")

# # garante tipos alinhados
# rad_delta_df["pair"] = rad_delta_df["pair"].astype(str)
# disp_df2["pair"] = disp_df2["pair"].astype(str)

# # merge 1:1 esperado
# merged_all = disp_df2.merge(
#     rad_delta_df,
#     on=keys,
#     how="inner",
#     suffixes=("", "_rad"),
#     validate="one_to_one",
# )

# print(f"Linhas displacement: {len(disp_df2)}")
# print(f"Linhas radiomics deltas: {len(rad_delta_df)}")
# print(f"Linhas após merge (inner): {len(merged_all)}")

# # Anexa covariáveis/infos técnicas POR IMAGEM de referência (ID_IMG_ref)
# # Cada linha do delta (pair) referencia uma imagem "alvo":
# # - pair 12: i2 (follow-up)
# # - pair 13: i3
# # - pair 23: i3
# merged_all["pair"] = merged_all["pair"].astype(str)
# merged_all["ID_IMG_ref"] = np.select(
#     [merged_all["pair"].eq("12"), merged_all["pair"].eq("13"), merged_all["pair"].eq("23")],
#     [merged_all["ID_IMG_i2"], merged_all["ID_IMG_i3"], merged_all["ID_IMG_i3"]],
#     default=merged_all["ID_IMG_i2"],
# )
# merged_all["ID_IMG_ref"] = merged_all["ID_IMG_ref"].astype(str).str.strip()

# # Remove AGE/SEX/DIAG vindos do pipeline anterior (triplet/i1). O merge com adnimerged
# # traz os valores por ID_IMG_ref; sem esse drop o pandas duplica colunas -> SEX_x / SEX_y.
# _drop_prev = [c for c in ("AGE", "DIAG", "SEX") if c in merged_all.columns]
# if _drop_prev:
#     merged_all = merged_all.drop(columns=_drop_prev)

# tech_df = pd.read_csv("/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv")
# tech_cols = [
#     "ID_IMG",
#     "AGE",
#     "DIAG",
#     "SEX",
#     "FIELD_STRENGTH",
#     "SLICE_THICKNESS",
#     "MANUFACTURER",
#     "MFG_MODEL",
# ]
# tech_sub = tech_df.loc[:, tech_cols].copy()
# tech_sub["ID_IMG"] = tech_sub["ID_IMG"].astype(str).str.strip()
# tech_sub = tech_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

# merged_all = merged_all.merge(
#     tech_sub.rename(columns={"ID_IMG": "ID_IMG_ref"}),
#     on="ID_IMG_ref",
#     how="left",
#     validate="many_to_one",
# )

# # Opcional: batch pronto para neuroCombat
# merged_all["batch"] = (
#     merged_all[["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH", "SLICE_THICKNESS"]]
#     .astype(str)
#     .agg("_".join, axis=1)
# )

# # Reorganiza colunas no merged final: chaves -> covariáveis/tempos -> técnicas -> features
# key_cols = keys
# extra_cols = ["t12", "t13", "t23", "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG"]
# technical_cols = ["ID_IMG_ref", "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL", "batch"]

# # separa features (tudo que não é chave nem extra nem técnico)
# feature_cols = [c for c in merged_all.columns if c not in set(key_cols + extra_cols + technical_cols)]

# ordered = [c for c in key_cols if c in merged_all.columns]
# ordered += [c for c in extra_cols if c in merged_all.columns]
# ordered += [c for c in technical_cols if c in merged_all.columns]
# ordered += feature_cols
# merged_all = merged_all.loc[:, ordered]

# out_merged_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features.csv"
# merged_all.to_csv(out_merged_path, index=False)

# out_merged_path

# Harmonização entre scanners delta atributos

In [29]:
# # Harmonização com NeuroCombat (ComBat multi-site) — etapa final deste notebook.
# # Próximo passo (modelagem): z-score com sklearn StandardScaler dentro de Pipeline + CV por fold,
# # usando o CSV harmonizado como entrada (evita vazamento entre treino/validação).

# # Entrada: all_delta_features.csv (mesmo `ab` das células anteriores).
# # Saída: colunas de metadados inalteradas + features ajustadas por scanner (`batch`).

# import numpy as np
# import pandas as pd

# from neuroCombat import neuroCombat

# in_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features.csv"
# out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features_neurocombat.csv"

# key_cols = [
#     "ID_PT",
#     "COMBINATION_NUMBER",
#     "TRIPLET_IDX",
#     "pair",
#     "ID_IMG_i1",
#     "ID_IMG_i2",
#     "ID_IMG_i3",
#     "roi",
#     "side",
#     "label",
# ]
# extra_cols = ["t12", "t13", "t23", "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG"]
# technical_cols = ["ID_IMG_ref", "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL", "batch"]

# meta_cols = [c for c in key_cols + extra_cols + technical_cols]

# df = pd.read_csv(in_path)
# missing_meta = [c for c in meta_cols if c not in df.columns]
# if missing_meta:
#     raise ValueError(f"CSV sem colunas esperadas: {missing_meta}")

# feature_cols = [c for c in df.columns if c not in meta_cols]
# if not feature_cols:
#     raise ValueError("Nenhuma coluna numérica (feature) encontrada além dos metadados.")

# X = df[feature_cols].apply(pd.to_numeric, errors="coerce")

# # Covariáveis biológicas preservadas no modelo além do batch (NeuroCombat usa design matrix)
# covar_cols = ["batch", "AGE", "SEX", "DIAG"]
# for c in covar_cols:
#     if c not in df.columns:
#         raise ValueError(f"Coluna necessária ausente: {c}")

# covars = df[covar_cols].copy()
# covars["AGE"] = pd.to_numeric(covars["AGE"], errors="coerce")

# mask = covars["batch"].notna().astype(bool)
# mask &= X.notna().all(axis=1)
# mask &= covars["AGE"].notna()
# mask &= covars["SEX"].notna()
# mask &= covars["DIAG"].notna()

# n_drop = int((~mask).sum())
# if n_drop:
#     print(f"Linhas excluídas (batch/features/covars incompletos): {n_drop}")

# df_fit = df.loc[mask].reset_index(drop=True)
# covars_fit = covars.loc[mask].reset_index(drop=True)
# X_fit = X.loc[mask].reset_index(drop=True)

# # neuroCombat espera dat com shape (n_features, n_amostras)
# dat = X_fit.to_numpy(dtype=float).T

# n_batch = covars_fit["batch"].nunique()
# print(f"Amostras na harmonização: {dat.shape[1]} | Features: {dat.shape[0]} | Níveis de batch: {n_batch}")
# if n_batch < 2:
#     raise ValueError(
#         "NeuroCombat requer pelo menos 2 batches distintos. Verifique a coluna batch."
#     )

# res = neuroCombat(
#     dat=dat,
#     covars=covars_fit,
#     batch_col="batch",
#     categorical_cols=["SEX", "DIAG"],
#     eb=True,
#     parametric=True,
#     mean_only=False,
# )

# harm = res["data"].T  # volta para (n_amostras, n_features)
# harm_df = pd.DataFrame(harm, columns=feature_cols, index=df_fit.index)

# out = pd.concat([df_fit[meta_cols], harm_df], axis=1)
# # mantém a mesma ordem de colunas do arquivo de entrada (features harmonizadas no lugar das originais)
# out = out.loc[:, df.columns]

# out.to_csv(out_path, index=False)
# print(f"Salvo: {out_path}")

# out_path

# Harmonização entre scanners atributos unitários

In [30]:
# # Harmonização entre scanners (NeuroCombat) — atributos unitários por conjunto (long)
# #
# # Entrada:  all_unitary_features.csv
# # Saída:    all_unitary_features_neurocombat.csv

# import numpy as np
# import pandas as pd
# from neuroCombat import neuroCombat

# in_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_unitary_features.csv"
# out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_unitary_features_neurocombat.csv"

# df_in = pd.read_csv(in_path)

# # não manter MRI_DATE no unitário harmonizado
# if "MRI_DATE" in df_in.columns:
#     df_in = df_in.drop(columns=["MRI_DATE"])

# key_cols = [
#     "ID_PT",
#     "COMBINATION_NUMBER",
#     "TRIPLET_IDX",
#     "pair",
#     "ID_IMG_i1",
#     "ID_IMG_i2",
#     "ID_IMG_i3",
#     "ref_tag",
#     "roi",
#     "side",
#     "label",
# ]
# extra_cols = ["t12", "t13", "t23", "GROUP", "SEX", "DIAG", "AGE", "TIME_PROG"]
# technical_cols = [
#     "ID_IMG_ref",
#     "FIELD_STRENGTH",
#     "SLICE_THICKNESS",
#     "MANUFACTURER",
#     "MFG_MODEL",
#     "batch",
# ]

# meta_cols = [c for c in key_cols + extra_cols + technical_cols if c in df_in.columns]

# missing_meta = [c for c in (key_cols + extra_cols + technical_cols) if c not in df_in.columns]
# if missing_meta:
#     raise ValueError(f"CSV sem colunas esperadas: {missing_meta}")

# feature_cols = [c for c in df_in.columns if c not in meta_cols]
# if not feature_cols:
#     raise ValueError("Nenhuma coluna numérica (feature) encontrada além dos metadados.")

# X = df_in[feature_cols].apply(pd.to_numeric, errors="coerce")

# # covariáveis biológicas preservadas no modelo além do batch
# covar_cols = ["batch", "AGE", "SEX", "DIAG", "TIME_PROG", "GROUP"]
# for c in covar_cols:
#     if c not in df_in.columns:
#         raise ValueError(f"Coluna necessária ausente: {c}")

# covars = df_in[covar_cols].copy()
# covars["AGE"] = pd.to_numeric(covars["AGE"], errors="coerce")
# covars["TIME_PROG"] = pd.to_numeric(covars["TIME_PROG"], errors="coerce")

# mask = covars["batch"].notna().astype(bool)
# mask &= X.notna().all(axis=1)
# mask &= covars["AGE"].notna()
# mask &= covars["TIME_PROG"].notna()
# mask &= covars["SEX"].notna()
# mask &= covars["DIAG"].notna()
# mask &= covars["GROUP"].notna()

# n_drop = int((~mask).sum())
# if n_drop:
#     print(f"Linhas excluídas (batch/features/covars incompletos): {n_drop}")

# df_fit = df_in.loc[mask].reset_index(drop=True)
# covars_fit = covars.loc[mask].reset_index(drop=True)
# X_fit = X.loc[mask].reset_index(drop=True)

# # neuroCombat espera dat com shape (n_features, n_amostras)
# dat = X_fit.to_numpy(dtype=float).T

# n_batch = covars_fit["batch"].nunique()
# print(f"Amostras na harmonização: {dat.shape[1]} | Features: {dat.shape[0]} | Níveis de batch: {n_batch}")
# if n_batch < 2:
#     raise ValueError("NeuroCombat requer pelo menos 2 batches distintos. Verifique a coluna batch.")

# res = neuroCombat(
#     dat=dat,
#     covars=covars_fit,
#     batch_col="batch",
#     categorical_cols=["SEX", "DIAG", "GROUP"],
#     eb=True,
#     parametric=True,
#     mean_only=False,
# )

# harm = res["data"].T
# harm_df = pd.DataFrame(harm, columns=feature_cols, index=df_fit.index)

# out = pd.concat([df_fit[meta_cols], harm_df], axis=1)

# # mantém a ordem de colunas do arquivo de entrada (sem MRI_DATE)
# out = out.loc[:, df_in.columns]

# out.to_csv(out_path, index=False)
# print(f"Salvo: {out_path}")

# out_path


# Torna planilhas wide

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Torna planilhas WIDE (triplet)
# 1 linha = (ID_PT, COMBINATION_NUMBER, TRIPLET_IDX, roi, side)
# - unitary: pivot por imagem de referência (i1/i2/i3)
# - delta:   pivot por par (12/13/23)
# Saídas: *_wide.csv (e opcionalmente *_wide_merged.csv)
# -----------------------------

# ab = "abordagem_1_sMCI_pMCI"
base_dir = "/mnt/study-data/pgirardi/graphs/csvs"

unitary_path = f"{base_dir}/{ab}/all_unitary_features_neurocombat.csv"
delta_path = f"{base_dir}/{ab}/all_delta_features_neurocombat.csv"

out_unitary_wide = f"{base_dir}/{ab}/all_unitary_features_neurocombat_wide.csv"
out_delta_wide = f"{base_dir}/{ab}/all_delta_features_neurocombat_wide.csv"
out_merged_wide = f"{base_dir}/{ab}/all_features_neurocombat_wide.csv"

# Chave do "exemplo" (ROI/hemisfério dentro do triplet)
key_cols = ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "roi", "side"]

# -----------------------------
# UNITÁRIOS -> WIDE (i1/i2/i3)
# -----------------------------
unit_df = pd.read_csv(unitary_path)

required_u = set(key_cols + ["ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3", "ID_IMG_ref"])
missing_u = sorted(required_u - set(unit_df.columns))
if missing_u:
    raise ValueError(f"Colunas ausentes em unitary: {missing_u}")

# Descobre se cada linha pertence ao i1/i2/i3 via ID_IMG_ref
# (np.select precisa que choices e default tenham dtype compatível)
ref = unit_df["ID_IMG_ref"].astype(str)
unit_df["ref_pos"] = np.select(
    [
        ref == unit_df["ID_IMG_i1"].astype(str),
        ref == unit_df["ID_IMG_i2"].astype(str),
        ref == unit_df["ID_IMG_i3"].astype(str),
    ],
    ["i1", "i2", "i3"],
    default="unknown",
)

bad_ref = (unit_df["ref_pos"] == "unknown").sum()
if bad_ref:
    raise ValueError(
        f"{bad_ref} linhas unitary com ID_IMG_ref que não bate com i1/i2/i3. "
        "Verifique colunas ID_IMG_i1/i2/i3 e ID_IMG_ref."
    )

# Para a modelagem, vamos manter o target/identificadores fora das features
# (você pode adicionar covariáveis depois via ColumnTransformer)
# Mantemos SEX como metadado para permitir balanceamento por GROUP x SEX.
meta_keep_u = key_cols + ["GROUP", "SEX"]
for c in meta_keep_u:
    if c not in unit_df.columns:
        raise ValueError(f"Coluna esperada não encontrada em unitary: {c}")

# Colunas que nunca devem virar feature
never_feature_u = set(
    meta_keep_u
    + [
        "ref_pos",
        "ID_IMG_i1",
        "ID_IMG_i2",
        "ID_IMG_i3",
        "ID_IMG_ref",
        "ref_tag",
        "pair",
        "t12",
        "t13",
        "t23",
        "DIAG",
        "AGE",
        "TIME_PROG",
        "FIELD_STRENGTH",
        "SLICE_THICKNESS",
        "MANUFACTURER",
        "MFG_MODEL",
        "batch",
        "label",  # label aqui é código ROI+side, não target
    ]
)

feature_cols_u = [c for c in unit_df.columns if c not in never_feature_u]

# Garante que features são numéricas quando possível
for c in feature_cols_u:
    unit_df[c] = pd.to_numeric(unit_df[c], errors="coerce")

# Pivot: index = chave + GROUP ; columns = ref_pos (i1/i2/i3)
wide_u = unit_df[meta_keep_u + ["ref_pos"] + feature_cols_u].pivot_table(
    index=meta_keep_u,
    columns="ref_pos",
    values=feature_cols_u,
    aggfunc="first",
)

# Flatten multiindex: feat__i1
wide_u.columns = [f"{feat}__{pos}" for feat, pos in wide_u.columns.to_list()]
wide_u = wide_u.reset_index()

# -----------------------------
# DELTAS -> WIDE (d12/d13/d23)
# -----------------------------
delta_df = pd.read_csv(delta_path)

required_d = set(key_cols + ["pair", "GROUP", "SEX"])
missing_d = sorted(required_d - set(delta_df.columns))
if missing_d:
    raise ValueError(f"Colunas ausentes em delta: {missing_d}")

# Normaliza par como string ("12", "13", "23")
delta_df["pair"] = delta_df["pair"].astype(str).str.strip()

# Mantemos SEX como metadado para permitir balanceamento por GROUP x SEX.
meta_keep_d = key_cols + ["GROUP", "SEX"]

never_feature_d = set(
    meta_keep_d
    + [
        "pair",
        "ID_IMG_i1",
        "ID_IMG_i2",
        "ID_IMG_i3",
        "ID_IMG_ref",
        "DIAG",
        "AGE",
        "TIME_PROG",
        "FIELD_STRENGTH",
        "SLICE_THICKNESS",
        "MANUFACTURER",
        "MFG_MODEL",
        "batch",
        "label",  # código ROI+side
        "t12",
        "t13",
        "t23",
    ]
)

feature_cols_d = [c for c in delta_df.columns if c not in never_feature_d]

for c in feature_cols_d:
    delta_df[c] = pd.to_numeric(delta_df[c], errors="coerce")

wide_d = delta_df[meta_keep_d + ["pair"] + feature_cols_d].pivot_table(
    index=meta_keep_d,
    columns="pair",
    values=feature_cols_d,
    aggfunc="first",
)

# feat__d12
wide_d.columns = [f"{feat}__d{pair}" for feat, pair in wide_d.columns.to_list()]
wide_d = wide_d.reset_index()

# -----------------------------
# Salva
# -----------------------------
wide_u.to_csv(out_unitary_wide, index=False)
wide_d.to_csv(out_delta_wide, index=False)

# # Merge opcional (unitários + deltas) na mesma linha
# wide_merged = wide_u.merge(wide_d, on=meta_keep_u, how="inner", validate="one_to_one")
# wide_merged.to_csv(out_merged_wide, index=False)

print("Unitary wide:", out_unitary_wide, wide_u.shape)
print("Delta wide:", out_delta_wide, wide_d.shape)
# print("Merged wide:", out_merged_wide, wide_merged.shape)

# Observação: o target para modelagem deve vir de GROUP (binário),
# e a seleção/normalização devem ser feitas apenas no fold de treino.


/mnt/study-data/pgirardi/tmp/ipykernel_3737319/2817636430.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  unit_df["ref_pos"] = np.select(
/mnt/study-data/pgirardi/tmp/ipykernel_3737319/2817636430.py:105: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide_u = wide_u.reset_index()
/mnt/study-data/pgirardi/tmp/ipykernel_3737319/2817636430.py:105: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once

# Seleção de atributos



In [ ]:
# dado o seguinte csv "/mnt/study-data/pgirardi/graphs/csvs/abordagem_teste/all_delta_features_neurocombat.csv", verifique se meu designe a seguir está coerente para que eu consiga iniciar a modelagem? 

# 1. Da maneira que está cada 3 linhas representam um conjunto com 3 imagens, sendo 20 regiões de interesse (coluna roi + side) vezes 3 pares de imagens (coluna pair), para cada linha representar um conjunto com 3 imagens longitudinais eu preciso realizar um flatten (concatenar as informações a cada tres linhas ou mesmo ID_PT + COMBINATION_NUMBER + TRIPLET_IDX) para que os atributos dos pares de imagens fiquem na sequencia na mesma linha e salvar esse novo arquivo que será a entrada dos modelos.

# 2. Com um arquivo csv com todos os atributos, em que cada linha representa um conjunto com 3 imagens, iniciar a etapa de balancemento dos dados em nível de paciente, para a classe sMCI e pMCI (coluna GROUP) e sexo F e M (coluna SEX) com downsampling.  

# 3. Após o balanceamento, realizar a selação de atributos, a priori em dois niveis, sendo 1. para filtrar atributos com alta correlação (features vs features), e 2. para filtrar atributos com baixa variância (features vs variância) e de alguma maneira printar a quantidade de atributos após as etapas e plotar isso para documentação. Em seguida (caso essas duas abordagens não sejam suficientemente robustas), utilizar o método selectkbest (univariado) do sklearn para mais uma etapa de filtragem e novamente de alguma maneira printar a quantidade de atributos após as etapas e plotar isso para documentação, e em seguida, caso ainda não seja suficiente, utilizar o método SFS sequential feature selector (multivariado) do sklearn para finalizar a seleção de atributos.  Finalmente, salvar o arquivo com os atributos filtrados. 

# 4. Realizar a etapa se separação dos folds de treino, validação e teste, tendo o cuidado pra não ter conjuntos (linhas) do mesmo paciente (coluna ID_PT) do treinamento no teste para evitar vazameto de dados (leakage).  Também para evitar vazamento de dados, aplicar a normalização z-score apenas no fold de treinamento, e a média e desvio padrão obtidos do treino aplicar no fold de validação e teste. 

# 5. iniciar de fato a modelagem para classificação supervisionada. 